In [2]:
import duckdb
import pandas as pd

con = duckdb.connect('../olist.duckdb')

In [ ]:
## Row count of each table
con.sql("""
    SELECT table_name, estimated_size AS row_count
    FROM duckdb_tables()
    ORDER BY row_count DESC
""").df()

,table_name,row_count
0,geolocation,1000163
1,order_items,112650
2,order_payments,103886
3,customers,99441
4,orders,99441
5,order_reviews,99224
6,products,32951
7,sellers,3095
8,product_category_name_translation,71


The dataset contains 9 tables. Row counts were retrieved via duckdb_tables() metadata rather than individual COUNT(*) calls, verified identical here since tables were bulk-loaded and never modified.

In [ ]:
## Date range of the dataset
con.sql("""
    SELECT 
      MIN(order_purchase_timestamp) AS first_order,
      MAX(order_purchase_timestamp) AS last_order
    FROM orders
""").df()

,first_order,last_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


The dataset spans September 4, 2016 to October 17, 2018 — about 25 months.

In [5]:
## Monthly order distribution
con.sql("""
    SELECT 
      date_trunc('month', order_purchase_timestamp) AS month,
      COUNT(order_id) AS n_orders
    FROM orders
    GROUP BY month
    ORDER BY month
""").df()

,month,n_orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


Order volumes are erratic in Sept–Nov 2016 (as low as 1 order in a month), consistent with a platform ramp-up or test period. The final months (Sept–Oct 2018) are also abnormally low, since October is cut off mid-month. Both edges are excluded from analysis. See Conclusions below.

In [6]:
## Order status breakdown
con.sql("""
    SELECT 
      order_status,
      COUNT(order_id) AS n_orders
    FROM orders
    GROUP BY order_status
    ORDER BY n_orders DESC
""").df()

,order_status,n_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


delivered accounts for over 97% of all orders and represents genuine completed purchases. Other statuses (canceled, unavailable, shipped, etc.) reflect orders that never reached the customer or are still in progress.

In [7]:
## Impact check: delivered vs delivered+shipped

con.sql("""
    SELECT COUNT(order_id) AS n_orders
    FROM orders
    WHERE order_status IN ('delivered', 'shipped')
""").df()

,n_orders
0,97585


Including shipped alongside delivered adds only +1.14% to the order count is negligible. delivered alone is used going forward for simplicity.

In [8]:
## customer_id vs customer_unique_id

con.sql("""
    SELECT 
      COUNT(DISTINCT customer_id) AS n_customer_id,
      COUNT(DISTINCT customer_unique_id) AS n_customer_unique_id
    FROM customers
""").df()

,n_customer_id,n_customer_unique_id
0,99441,96096


customer_id count exceeds customer_unique_id count, since a new customer_id is generated for every order, even for a returning customer. customer_unique_id is the true person-level identifier.

In [9]:
## Proof: one customer, multiple customer_id

con.sql("""
    SELECT customer_unique_id, COUNT(DISTINCT customer_id) AS n_customer_ids
    FROM customers
    GROUP BY customer_unique_id
    HAVING COUNT(DISTINCT customer_id) > 1
    ORDER BY n_customer_ids DESC
    LIMIT 10
""").df()

,customer_unique_id,n_customer_ids
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,6469f99c1f9dfae7733b25662e7f1782,7
3,1b6c7548a2a1f9037c1fd3ddfed95f33,7
4,ca77025e7201e3b30c44b472ff346268,7
5,47c1a3033b8b77b3ab6e109eb4d5fdf3,6
6,63cfc61cee11cbe306bff5857d00bfe4,6
7,12f5d6e1cbf93dafd9dcc19095df0b3d,6
8,f0e310a6839dce9de1638e0fe5ab282a,6
9,de34b16117594161a6a89c50b289d35a,6


Some customer_unique_id values are linked to multiple customer_id values, confirming that the same person can generate several order-level IDs across purchases.

In [10]:
## Repeat purchase rate 

con.sql("""
    WITH 
    customers_reorder AS (
      SELECT customer_unique_id
      FROM customers
      GROUP BY customer_unique_id
      HAVING COUNT(DISTINCT customer_id) > 1
    ),
    totals AS (
      SELECT 
        (SELECT COUNT(*) FROM customers_reorder) AS n_repeats,
        (SELECT COUNT(DISTINCT customer_unique_id) FROM customers) AS n_total
    )
    SELECT 
      n_repeats,
      n_total,
      n_repeats * 100.0 / n_total AS repeat_purchase_rate_pct
    FROM totals
""").df()

,n_repeats,n_total,repeat_purchase_rate_pct
0,2997,96096,3.118756


Only ~3% of customers made more than one purchase, which is notably low compared to typical e-commerce retention benchmarks (often 20-30%+), signaling a significant retention challenge for Olist.

In [11]:
## Orphan items check: items that are not in any order

con.sql("""
    SELECT oi.*
    FROM order_items oi
    LEFT JOIN orders o ON oi.order_id = o.order_id
    WHERE o.order_id IS NULL
""").df()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


No items exist without a matching order, referential integrity holds in this direction.

In [12]:
## Orders with no items

con.sql("""
    SELECT o.*
    FROM orders o
    LEFT JOIN order_items oi ON o.order_id = oi.order_id
    WHERE oi.order_id IS NULL
""").df()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,8e24261a7e58791d10cb1bf9da94df5c,64a254d30eed42cd0e6c36dddb88adf0,unavailable,2017-11-16 15:09:28,2017-11-16 15:26:57,NaT,NaT,2017-12-05
1,c272bcd21c287498b4883c7512019702,9582c5bbecc65eb568e2c1d839b5cba1,unavailable,2018-01-31 11:31:37,2018-01-31 14:23:50,NaT,NaT,2018-02-16
2,37553832a3a89c9b2db59701c357ca67,7607cd563696c27ede287e515812d528,unavailable,2017-08-14 17:38:02,2017-08-17 00:15:18,NaT,NaT,2017-09-05
3,d57e15fb07fd180f06ab3926b39edcd2,470b93b3f1cde85550fc74cd3a476c78,unavailable,2018-01-08 19:39:03,2018-01-09 07:26:08,NaT,NaT,2018-02-06
4,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaT,NaT,NaT,2018-09-12
...,...,...,...,...,...,...,...,...
770,aaab15da689073f8f9aa978a390a69d1,df20748206e4b865b2f14a5eabbfcf34,unavailable,2018-01-16 14:27:59,2018-01-17 03:37:34,NaT,NaT,2018-02-06
771,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaT,NaT,NaT,2018-10-01
772,a89abace0dcc01eeb267a9660b5ac126,2f0524a7b1b3845a1a57fcf3910c4333,canceled,2018-09-06 18:45:47,NaT,NaT,NaT,2018-09-27
773,a69ba794cc7deb415c3e15a0a3877e69,726f0894b5becdf952ea537d5266e543,unavailable,2017-08-23 16:28:04,2017-08-28 15:44:47,NaT,NaT,2017-09-15


775 orders have no matching items. These are expected to correspond mostly to canceled or unavailable orders, which never reached a confirmed cart, and are therefore already excluded by the delivered-only filter.

In [13]:
## Reviews per order distribution

con.sql("""
    WITH reviews_per_order AS (
      SELECT o.order_id, COUNT(r.review_id) AS n_reviews
      FROM orders o
      LEFT JOIN order_reviews r ON o.order_id = r.order_id
      GROUP BY o.order_id
    )
    SELECT n_reviews, COUNT(*) AS n_orders
    FROM reviews_per_order
    GROUP BY n_reviews
    ORDER BY n_reviews
""").df()

,n_reviews,n_orders
0,0,768
1,1,98126
2,2,543
3,3,4


Most orders (98,126) have exactly one review; 768 have none and a small number (547) have 2–3 reviews. This confirms order_reviews can have multiple rows per order and must be aggregated to order-level before joining.

## Conclusions & Analysis Scope

This exploration surfaced several data quality considerations that shape 
the rest of this project:

**Time period**: Sept–Nov 2016 (erratic volumes, likely platform ramp-up) 
and Sept–Oct 2018 (incomplete month) are excluded. 
→ **Analysis period: January 2017 – August 2018.**

**Order status**: Only `delivered` orders represent completed purchases. 
`shipped` adds a negligible +1.14% and is excluded for simplicity.
→ **Status filter: `delivered` only.**

**Customer identifier**: `customer_id` is generated per order, while 
`customer_unique_id` identifies the actual person. Using `customer_id` 
alone would artificially inflate the customer count and hide repeat 
purchases.
→ **Always use `customer_unique_id` for customer-level analysis.**

**Table grains**: `order_items`, `payments`, and `order_reviews` can all 
have multiple rows per order. They must be aggregated to order-level 
before joining with `orders` to avoid row duplication.

**Key finding**: only ~3% of customers made more than one purchase, a 
strong signal of a retention challenge, motivating the segmentation and 
CRM reactivation strategy explored in the next notebooks.

These rules will be centralized in the dbt staging layer (Phase 2) so 
they're applied consistently across all downstream analysis.